<a href="https://colab.research.google.com/github/Du-nara/ME421-Mechanical-Systems-Lab-A3/blob/main/Controls/E_20_249Controls.ipynbme421_rigidbodycontrolsystems_lab_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instructions

* Complete the activities in groups that were assigned for ME421 for the vibrations lab.

* Make a copy of this and save it in your group github group repository with your index number as the file name

* Do all your work, EXCLUSIVELY, in that saved notebook. Your github commits will serve as a reflection of your individual contributions.

# References

* https://github.com/mugalan/intrinsic-rigid-body-control-estimation/tree/main/rigid-body-control

## Setup: Install & Import Libraries

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
from scipy.linalg import expm
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

# Task #1

## Twin Rotor System Dynamic Model

### Derivation

**System Description:**
The twin rotor system consists of a rigid body (rotor assembly) that can rotate in 3D space. Two rotors generate forces:
- Rotor 1 (pitch): thrust along $\mathbf{b}_1$, moment about $\mathbf{b}_3$ axis, with angle $\alpha$
- Rotor 2 (yaw): thrust along $\mathbf{b}_1$, moment about $\mathbf{b}_3$ axis, with angle $\beta$

The rotation matrix $R \in SO(3)$ maps body-frame vectors to spatial (inertial) frame.

**Newton-Euler equations on $SO(3)$:**

The kinematics of the rotation matrix are:
$$\dot{R} = \hat{\omega} R$$
where $\hat{\omega}$ is the skew-symmetric matrix of the spatial angular velocity $\omega$.

The spatial angular momentum is:
$$\pi = R \mathbb{I} R^T \omega = R \mathbb{I} \Omega$$
where $\Omega = R^T \omega$ is the body-frame angular velocity, and $\mathbb{I}$ is the body-frame inertia tensor.

**Momentum dynamics:**

Differentiating $\pi$:
$$\dot{\pi} = \frac{d}{dt}(R \mathbb{I} \Omega) = \dot{R}\mathbb{I}\Omega + R\mathbb{I}\dot{\Omega}$$

By Euler's equation in the body frame:
$$\mathbb{I}\dot{\Omega} = -\Omega \times \mathbb{I}\Omega + \tau^{\text{body}}$$

So the spatial momentum equation becomes:
$$\dot{\pi} = \tau^e + \tau^u$$

with the recovery: $\omega = (\mathbb{I}^R)^{-1}\pi$ where $\mathbb{I}^R = R\mathbb{I}R^T$ is the spatially-resolved inertia.

**Control torque:**
$$\tau^u = R \begin{bmatrix}1 & 0\\ 0 & 0\\ 0 & 1\end{bmatrix} \begin{bmatrix}\cos\alpha & -\cos\beta\\ \sin\alpha & -\sin\beta\end{bmatrix} \begin{bmatrix}u_1\\ u_2\end{bmatrix}$$

This maps the two rotor thrust inputs $u_1, u_2$ to torques in the $\mathbf{b}_1$ and $\mathbf{b}_3$ directions.

**Constraint torque** (prevents rotation about $\mathbf{b}_2$):
$$\tau^e = R\begin{bmatrix}0\\T_2\\0\end{bmatrix}$$

The constraint force $T_2$ is determined by the constraint that the $\mathbf{b}_2$ component of angular acceleration is zero:
$$T_2 = e_2^T\left(\Omega \times \mathbb{I}\Omega + \mathbb{I}\dot{\Omega}\right)$$

This is solved implicitly from the equations of motion, ensuring the system respects its mechanical constraint.

**Summary — the twin rotor model:**
$$\boxed{\dot{R} = \hat{\omega}R, \qquad \dot{\pi} = \tau^e + \tau^u, \qquad \omega = (\mathbb{I}^R)^{-1}\pi}$$

## Helper Functions: SO(3) Utilities

In [ ]:
# ============================================================
# SO(3) UTILITY FUNCTIONS
# ============================================================

def hat(v):
    """Skew-symmetric matrix (hat map) for vector v in R^3."""
    v = np.asarray(v).flatten()
    return np.array([
        [ 0,    -v[2],  v[1]],
        [ v[2],  0,    -v[0]],
        [-v[1],  v[0],  0   ]
    ])

def vee(S):
    """Inverse of hat map: extract vector from skew-symmetric matrix."""
    return np.array([S[2,1], S[0,2], S[1,0]])

def project_SO3(R):
    """Project matrix onto SO(3) using SVD to avoid numerical drift."""
    U, _, Vt = np.linalg.svd(R)
    D = np.diag([1, 1, np.linalg.det(U @ Vt)])
    return U @ D @ Vt

def cross(a, b):
    """Cross product a × b."""
    return np.cross(a, b)

def Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

def Ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]])

def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

print('SO(3) utility functions defined.')

## System Parameters

In [ ]:
# ============================================================
# TWIN ROTOR SYSTEM PARAMETERS
# ============================================================

# Inertia tensor (body frame) — diagonal for a symmetric rotor assembly
# Units: kg·m²
I_xx = 0.05  # Pitch axis inertia
I_yy = 0.10  # (constrained) axis inertia
I_zz = 0.05  # Yaw axis inertia
I_body = np.diag([I_xx, I_yy, I_zz])
I_body_inv = np.linalg.inv(I_body)

# Rotor geometry angles (radians)
# alpha: pitch rotor angle from b1 axis
# beta:  yaw rotor angle from b1 axis
alpha = 0.0          # pitch rotor aligned with b1
beta  = np.pi / 2.0  # yaw rotor at 90 degrees

# Control input mapping matrix B (3x2)
# tau^u = R @ B_mat @ [u1; u2]
B_geom = np.array([[1, 0],
                   [0, 0],
                   [0, 1]])  # selects b1 and b3 axes

B_rot = np.array([[np.cos(alpha), -np.cos(beta)],
                  [np.sin(alpha), -np.sin(beta)]])

B_mat = B_geom @ B_rot  # 3x2 control input matrix

print('System parameters set.')
print(f'Inertia tensor:\n{I_body}')
print(f'Control input matrix B:\n{B_mat}')

## Equations of Motion

In [ ]:
# ============================================================
# EQUATIONS OF MOTION FOR TWIN ROTOR SYSTEM
# ============================================================

def compute_constraint_torque(R, Omega, u1, u2):
    """
    Compute the constraint torque T2 that enforces zero angular
    acceleration about the b2 axis.

    From the equation of motion in body frame:
        I * dOmega/dt = -Omega × I*Omega + tau_body

    The b2 component of tau_body comes only from T2.
    Control torque has NO b2 component (by design of B_mat).

    For constraint: e2^T (Omega × I*Omega + I*dOmega/dt) is solved
    with dOmega_2/dt = 0 (constraint).

    This gives:
        T2 = e2^T (Omega × I*Omega) - e2^T (I * (dOmega_1/dt * e1 + dOmega_3/dt * e3))

    However, for simulation we treat the system as if the constraint
    simply removes b2 degree of freedom and T2 maintains it.
    We compute T2 = e2^T(Omega × I*Omega + I*dOmega/dt) evaluated
    at dOmega_2 = 0.
    """
    IOmega = I_body @ Omega
    gyro   = cross(Omega, IOmega)    # Omega × I*Omega

    # Control torque in body frame
    u_vec = np.array([u1, u2])
    tau_u_body = B_mat @ u_vec       # 3-vector in body frame

    # For the unconstrained b1, b3 components:
    # I * dOmega/dt = -gyro + tau_u_body + [0, T2, 0]^T
    # For b2: I_yy * 0 = -gyro[1] + 0 + T2  =>  T2 = gyro[1]
    T2 = gyro[1]
    return T2


def twin_rotor_ode(t, state, u_func):
    """
    ODE for the twin rotor system.

    State vector (12 components):
        state[0:9]  = R flattened (row-major)
        state[9:12] = pi (spatial angular momentum)

    Returns d/dt(state).
    """
    R   = state[0:9].reshape(3, 3)
    pi  = state[9:12]

    # Spatially-resolved inertia
    IR  = R @ I_body @ R.T
    IR_inv = np.linalg.inv(IR)

    # Angular velocity
    omega = IR_inv @ pi

    # Body-frame angular velocity
    Omega = R.T @ omega

    # Control inputs
    u1, u2 = u_func(t, R, pi, omega)

    # Control torque in spatial frame
    u_vec   = np.array([u1, u2])
    tau_u   = R @ B_mat @ u_vec

    # Constraint torque (maintains b2 constraint)
    T2 = compute_constraint_torque(R, Omega, u1, u2)
    tau_e = R @ np.array([0.0, T2, 0.0])

    # Kinematics: dR/dt = hat(omega) @ R
    dR = hat(omega) @ R

    # Momentum dynamics: dpi/dt = tau_e + tau_u
    dpi = tau_e + tau_u

    d_state = np.concatenate([dR.flatten(), dpi])
    return d_state


def simulate(u_func, t_span, t_eval, R0=None, pi0=None):
    """
    Simulate the twin rotor system.

    Args:
        u_func:  callable u_func(t, R, pi, omega) -> (u1, u2)
        t_span:  (t0, tf)
        t_eval:  array of times at which to record state
        R0:      initial rotation matrix (default: identity)
        pi0:     initial spatial momentum (default: zero)

    Returns:
        t_eval, R_traj (N,3,3), omega_traj (N,3), pi_traj (N,3)
    """
    if R0 is None:
        R0 = np.eye(3)
    if pi0 is None:
        pi0 = np.zeros(3)

    state0 = np.concatenate([R0.flatten(), pi0])

    sol = solve_ivp(
        fun=lambda t, s: twin_rotor_ode(t, s, u_func),
        t_span=t_span,
        y0=state0,
        t_eval=t_eval,
        method='RK45',
        rtol=1e-8,
        atol=1e-10
    )

    N = len(sol.t)
    R_traj    = np.zeros((N, 3, 3))
    pi_traj   = np.zeros((N, 3))
    omega_traj = np.zeros((N, 3))

    for i in range(N):
        R_i  = sol.y[0:9, i].reshape(3, 3)
        R_i  = project_SO3(R_i)  # keep on SO(3)
        pi_i = sol.y[9:12, i]
        IR_i = R_i @ I_body @ R_i.T
        omega_i = np.linalg.inv(IR_i) @ pi_i

        R_traj[i]    = R_i
        pi_traj[i]   = pi_i
        omega_traj[i] = omega_i

    return sol.t, R_traj, omega_traj, pi_traj


print('Equations of motion defined.')

# Task #2 — Simulation and Animation of the Twin Rotor System

We simulate the system for two realistic input scenarios:
1. **Vertical axis spin** — input $u_2$ drives rotation about the vertical ($\mathbf{b}_3$) axis
2. **Pitch oscillation (swing up and down)** — input $u_1$ drives oscillation about the pitch ($\mathbf{b}_1$) axis

In [ ]:
# ============================================================
# TASK 2: SIMULATION SCENARIOS
# ============================================================

T_END = 10.0
dt    = 0.02
t_eval = np.arange(0, T_END + dt, dt)

# ----- Scenario 1: Vertical axis spin (yaw) -----
def u_yaw_spin(t, R, pi, omega):
    """Constant yaw torque to produce a vertical-axis spin."""
    u1 = 0.0
    u2 = 0.3  # Nm — yaw control torque
    return u1, u2

t1, R1, omega1, pi1 = simulate(
    u_yaw_spin,
    t_span=(0, T_END),
    t_eval=t_eval,
    R0=np.eye(3),
    pi0=np.zeros(3)
)
print('Scenario 1 (Yaw spin) simulated.')

# ----- Scenario 2: Pitch oscillation -----
def u_pitch_oscillate(t, R, pi, omega):
    """Sinusoidal pitch torque for swing-up/down motion."""
    u1 = 0.3 * np.sin(0.5 * np.pi * t)  # sinusoidal pitch
    u2 = 0.0
    return u1, u2

t2, R2, omega2, pi2 = simulate(
    u_pitch_oscillate,
    t_span=(0, T_END),
    t_eval=t_eval,
    R0=np.eye(3),
    pi0=np.zeros(3)
)
print('Scenario 2 (Pitch oscillation) simulated.')

In [ ]:
# ============================================================
# PLOTTING: Angular velocities over time
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
labels = ['ω₁ (roll)', 'ω₂ (pitch constraint)', 'ω₃ (yaw)']
colors = ['tab:blue', 'tab:orange', 'tab:green']
titles = ['Scenario 1: Yaw Spin', 'Scenario 2: Pitch Oscillation']
trajs  = [omega1, omega2]
times  = [t1, t2]

for row, (traj, tt, title) in enumerate(zip(trajs, times, titles)):
    for col, (lbl, clr) in enumerate(zip(labels, colors)):
        ax = axes[row, col]
        ax.plot(tt, traj[:, col], color=clr, linewidth=1.5)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('rad/s')
        ax.set_title(f'{title}\n{lbl}')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/task2_angular_velocities.png', dpi=120, bbox_inches='tight')
plt.show()
print('Angular velocity plots saved.')

In [ ]:
# ============================================================
# VISUALISATION: 3D body-frame axes trajectory
# ============================================================

def plot_orientation_trajectory(t, R_traj, title, filename):
    """
    Plot the tip of each body-frame axis (b1, b2, b3)
    as it sweeps through the unit sphere.
    """
    fig = plt.figure(figsize=(8, 7))
    ax  = fig.add_subplot(111, projection='3d')

    colors = ['red', 'green', 'blue']
    names  = ['b₁', 'b₂', 'b₃']

    for k, (clr, nm) in enumerate(zip(colors, names)):
        # R[:, k] is the k-th column = image of e_k in spatial frame
        xs = R_traj[:, 0, k]
        ys = R_traj[:, 1, k]
        zs = R_traj[:, 2, k]
        ax.plot(xs, ys, zs, color=clr, linewidth=1.2, label=nm)
        ax.scatter(xs[0],  ys[0],  zs[0],  color=clr, s=40, marker='o')
        ax.scatter(xs[-1], ys[-1], zs[-1], color=clr, s=60, marker='*')

    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    plt.savefig(filename, dpi=120, bbox_inches='tight')
    plt.show()

plot_orientation_trajectory(t1, R1,
    'Scenario 1 — Yaw Spin: Body Axes on Unit Sphere',
    '/tmp/task2_scenario1_sphere.png')

plot_orientation_trajectory(t2, R2,
    'Scenario 2 — Pitch Oscillation: Body Axes on Unit Sphere',
    '/tmp/task2_scenario2_sphere.png')

In [ ]:
# ============================================================
# ANIMATION: Twin rotor 3D body frame
# ============================================================

def animate_twin_rotor(t, R_traj, title='Twin Rotor Animation',
                       skip=5):
    """
    Create a 3D animation of the twin rotor body frame.
    Returns the animation object.
    """
    idx = np.arange(0, len(t), skip)

    fig = plt.figure(figsize=(7, 6))
    ax  = fig.add_subplot(111, projection='3d')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(-1.5, 1.5)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)

    colors = ['red', 'green', 'blue']
    names  = ['b₁', 'b₂', 'b₃']
    scale  = 1.0

    arrows = []
    for k, (clr, nm) in enumerate(zip(colors, names)):
        col = R_traj[0, :, k] * scale
        q = ax.quiver(0, 0, 0, col[0], col[1], col[2],
                      color=clr, arrow_length_ratio=0.15,
                      linewidth=2.5, label=nm)
        arrows.append(q)

    # Draw rotor arms (b1 arm and b3 arm)
    arm_len = 0.8
    arm1, = ax.plot([], [], [], 'k-', linewidth=3)
    arm2, = ax.plot([], [], [], 'gray', linewidth=3)
    time_text = ax.text2D(0.02, 0.95, '', transform=ax.transAxes)

    ax.legend(loc='upper right')

    def update(frame):
        i = idx[frame]
        R = R_traj[i]

        # Remove old arrows
        for arr in arrows:
            arr.remove()

        arrows.clear()
        for k, clr in enumerate(colors):
            col = R[:, k] * scale
            q = ax.quiver(0, 0, 0, col[0], col[1], col[2],
                          color=clr, arrow_length_ratio=0.15,
                          linewidth=2.5)
            arrows.append(q)

        # Arms
        b1 = R[:, 0] * arm_len
        b3 = R[:, 2] * arm_len
        arm1.set_data_3d([-b1[0], b1[0]], [-b1[1], b1[1]], [-b1[2], b1[2]])
        arm2.set_data_3d([-b3[0], b3[0]], [-b3[1], b3[1]], [-b3[2], b3[2]])

        time_text.set_text(f't = {t[i]:.2f}s')
        return arrows + [arm1, arm2, time_text]

    ani = FuncAnimation(
        fig, update,
        frames=len(idx),
        interval=50,
        blit=False
    )
    plt.close(fig)
    return ani


# Create animations
ani1 = animate_twin_rotor(t1, R1, title='Scenario 1: Yaw Spin', skip=3)
ani2 = animate_twin_rotor(t2, R2, title='Scenario 2: Pitch Oscillation', skip=3)

print('Animations created.')
print('To display in Colab, run: HTML(ani1.to_jshtml())')

In [ ]:
# Display Animation 1 — Yaw Spin
HTML(ani1.to_jshtml())

In [ ]:
# Display Animation 2 — Pitch Oscillation
HTML(ani2.to_jshtml())

# Task #3 — PID Tracking Controller on SO(3)

## Controller Derivation

**Setup:**
Let $R_r(t)$ be the desired (reference) trajectory with $\hat{\omega}_r = \dot{R}_r R_r^T$.

Define the **configuration error**:
$$R_e = R_r R^T$$

The **spatial angular velocity error** satisfies:
$$\hat{\omega}_e = \dot{R}_e R_e^T = \hat{\omega}_r - R_e \hat{\omega} R_e^T$$
$$\Rightarrow \omega_e = \omega_r - R_e \omega$$

The **angular momentum error** is:
$$\pi_e = R_e^T \pi_r - \pi$$

Its derivative (from the derivation in the lab sheet):
$$\dot{\pi}_e = (R\dot{\Pi}_r + \omega \times R_e^T \pi_r) - \tau^u - \tau^e$$

**PID Control Law:**

Choose the control torque to:
1. Cancel the feed-forward terms from the reference trajectory
2. Add PD feedback on the orientation error
3. Add integral action on the orientation error

The **orientation error vector** on SO(3) is extracted via the vee map:
$$e_R = \frac{1}{2}\text{vee}(R_e - R_e^T)$$

The PID control law:
$$\tau^u = R\dot{\Pi}_r + \omega \times R_e^T \pi_r + K_P e_R + K_D \omega_e + K_I \int_0^t e_R \, d\tau - \tau^e$$

where $K_P, K_D, K_I > 0$ are positive definite gain matrices.

Since the system has a 2D control input (only $u_1, u_2$), we use the pseudo-inverse of $B_{\text{mat}}$ to allocate the desired torque to the actuators:
$$\begin{bmatrix}u_1 \\ u_2\end{bmatrix} = (R B_{\text{mat}})^\dagger \tau^{\text{desired}}$$

**Note:** The constraint torque $\tau^e$ is automatically handled by the constraint mechanism — the controller only needs to produce torques in the controllable subspace.


In [ ]:
# ============================================================
# TASK 3: PID TRACKING CONTROLLER
# ============================================================

# --- PID Gains ---
Kp = 2.0   # Proportional gain (scalar for simplicity; use matrix for full DOF)
Kd = 1.5   # Derivative gain
Ki = 0.3   # Integral gain

# Integral state (global, reset each simulation)
integral_eR = np.zeros(3)

def reference_trajectory(t):
    """
    Define a smooth reference trajectory R_r(t) and omega_r(t).
    Example: sinusoidal rotation about the z-axis (yaw tracking).
    Returns R_r (3x3), omega_r (3,), d_omega_r (3,)
    """
    # Yaw reference: theta(t) = A*sin(freq*t)
    A    = np.pi / 4.0    # amplitude (45 degrees)
    freq = 0.5            # rad/s

    theta_r    =  A * np.sin(freq * t)
    dtheta_r   =  A * freq * np.cos(freq * t)
    ddtheta_r  = -A * freq**2 * np.sin(freq * t)

    R_r = Rz(theta_r)

    # omega_r such that hat(omega_r) = dR_r R_r^T
    # For Rz(theta): dRz/dt = dtheta/dt * hat(e3) @ Rz
    # => hat(omega_r) = dtheta/dt * hat(e3)  =>  omega_r = dtheta/dt * e3
    omega_r = np.array([0.0, 0.0, dtheta_r])

    # d_omega_r
    d_omega_r = np.array([0.0, 0.0, ddtheta_r])

    # Pi_r = I * Omega_r (body-frame reference momentum)
    Omega_r = R_r.T @ omega_r
    Pi_r    = I_body @ Omega_r

    # d_Pi_r (body-frame)
    d_Omega_r = R_r.T @ (d_omega_r - cross(omega_r, R_r @ Omega_r))
    d_Pi_r    = I_body @ d_Omega_r - cross(Omega_r, I_body @ Omega_r)

    # pi_r = R_r @ Pi_r  (spatial)
    pi_r    = R_r @ Pi_r
    dot_pi_r = R_r @ d_Pi_r + hat(omega_r) @ pi_r

    return R_r, omega_r, pi_r, dot_pi_r


def pid_controller(t, R, pi, omega, dt_ctrl=0.02):
    """
    PID tracking controller for twin rotor.
    Returns (u1, u2).
    """
    global integral_eR

    R_r, omega_r, pi_r, dot_pi_r = reference_trajectory(t)

    # Configuration error
    R_e = R_r @ R.T

    # Orientation error vector (vee of skew part of R_e)
    eR = 0.5 * vee(R_e - R_e.T)

    # Velocity error
    omega_e = omega_r - R_e @ omega

    # Integral update
    integral_eR += eR * dt_ctrl

    # Feed-forward + PD + I
    tau_desired = (dot_pi_r
                   + cross(omega, R_e.T @ pi_r)
                   + Kp * eR
                   + Kd * omega_e
                   + Ki * integral_eR)

    # Map desired torque to control inputs via pseudo-inverse
    # Effective input matrix in spatial frame: B_spatial = R @ B_mat
    B_spatial = R @ B_mat   # 3x2
    B_pinv    = np.linalg.pinv(B_spatial)  # 2x3

    u_vec = B_pinv @ tau_desired

    # Clamp to realistic actuator limits
    u_max = 2.0
    u_vec = np.clip(u_vec, -u_max, u_max)

    return float(u_vec[0]), float(u_vec[1])


print('PID controller defined.')

In [ ]:
# ============================================================
# SIMULATE WITH PID CONTROLLER
# ============================================================

# Reset integral state
integral_eR = np.zeros(3)

T_END_ctrl = 20.0
dt_ctrl    = 0.02
t_ctrl     = np.arange(0, T_END_ctrl + dt_ctrl, dt_ctrl)

# Initial condition: 20-degree yaw offset from reference
R0_ctrl  = Rz(np.radians(20))
pi0_ctrl = np.zeros(3)

t_c, R_c, omega_c, pi_c = simulate(
    pid_controller,
    t_span=(0, T_END_ctrl),
    t_eval=t_ctrl,
    R0=R0_ctrl,
    pi0=pi0_ctrl
)
print('PID simulation complete.')

In [ ]:
# ============================================================
# COMPUTE TRACKING ERROR
# ============================================================

eR_history  = np.zeros((len(t_c), 3))
eOm_history = np.zeros((len(t_c), 3))
theta_error = np.zeros(len(t_c))  # geodesic angle error

for i, ti in enumerate(t_c):
    R_r, omega_r, pi_r, _ = reference_trajectory(ti)
    R_e_i    = R_r @ R_c[i].T
    eR_i     = 0.5 * vee(R_e_i - R_e_i.T)
    omega_e_i = omega_r - R_e_i @ omega_c[i]

    eR_history[i]   = eR_i
    eOm_history[i]  = omega_e_i

    # Geodesic angle error: ||log(R_e)||_F / sqrt(2)
    trace_Re     = np.clip((np.trace(R_e_i) - 1.0) / 2.0, -1.0, 1.0)
    theta_error[i] = np.degrees(np.arccos(trace_Re))

print('Tracking errors computed.')

In [ ]:
# ============================================================
# PLOT TRACKING PERFORMANCE
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# 1. Geodesic orientation error
ax = axes[0]
ax.plot(t_c, theta_error, 'navy', linewidth=1.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (degrees)')
ax.set_title('Geodesic Orientation Tracking Error (degrees)')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

# 2. Orientation error components eR
ax = axes[1]
comp_labels = ['eR₁', 'eR₂', 'eR₃']
comp_colors = ['tab:red', 'tab:green', 'tab:blue']
for k, (lbl, clr) in enumerate(zip(comp_labels, comp_colors)):
    ax.plot(t_c, eR_history[:, k], color=clr, label=lbl, linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (rad)')
ax.set_title('Orientation Error Vector Components eR')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Angular velocity error
ax = axes[2]
om_labels = ['ωe₁', 'ωe₂', 'ωe₃']
for k, (lbl, clr) in enumerate(zip(om_labels, comp_colors)):
    ax.plot(t_c, eOm_history[:, k], color=clr, label=lbl, linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (rad/s)')
ax.set_title('Angular Velocity Tracking Error')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/task3_tracking_performance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Tracking performance plot saved.')

In [ ]:
# ============================================================
# COMPARE ACTUAL vs REFERENCE TRAJECTORY
# ============================================================

# Extract yaw angle from R_c (rotation about z-axis component)
yaw_actual    = np.array([np.arctan2(R_c[i, 1, 0], R_c[i, 0, 0]) for i in range(len(t_c))])
yaw_reference = np.array([np.arctan2(Rz(np.pi/4 * np.sin(0.5*ti))[1,0],
                                      Rz(np.pi/4 * np.sin(0.5*ti))[0,0]) for ti in t_c])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_c, np.degrees(yaw_reference), 'b--', linewidth=2, label='Reference yaw θ_r(t)')
ax.plot(t_c, np.degrees(yaw_actual),    'r-',  linewidth=1.5, label='Actual yaw θ(t)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Yaw angle (degrees)')
ax.set_title('PID Tracking: Reference vs Actual Yaw Angle')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/task3_yaw_tracking.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Final geodesic error: {theta_error[-1]:.4f} degrees')
print(f'Max geodesic error:   {theta_error.max():.4f} degrees')
print(f'Mean geodesic error:  {theta_error.mean():.4f} degrees')

In [ ]:
# ============================================================
# ANIMATE PID TRACKING
# ============================================================

def animate_tracking(t, R_actual, t_ref=None, title='PID Tracking', skip=4):
    """
    Animate actual vs reference body frames side-by-side.
    """
    idx = np.arange(0, len(t), skip)

    fig = plt.figure(figsize=(14, 6))
    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')

    for ax, ttl in [(ax1, 'Actual'), (ax2, 'Reference')]:
        ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(-1.5, 1.5)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        ax.set_title(ttl)

    colors  = ['red', 'green', 'blue']
    arrows1 = []
    arrows2 = []
    time_txt = ax1.text2D(0.02, 0.95, '', transform=ax1.transAxes)

    def make_arrows(ax, R, arrows_list):
        for arr in arrows_list:
            arr.remove()
        arrows_list.clear()
        for k, clr in enumerate(colors):
            col = R[:, k]
            q = ax.quiver(0,0,0, col[0],col[1],col[2],
                          color=clr, arrow_length_ratio=0.15, linewidth=2.5)
            arrows_list.append(q)

    # Initialise
    make_arrows(ax1, R_actual[0], arrows1)
    R_r0, *_ = reference_trajectory(t[0])
    make_arrows(ax2, R_r0, arrows2)

    def update(frame):
        i = idx[frame]
        make_arrows(ax1, R_actual[i], arrows1)
        R_ri, *_ = reference_trajectory(t[i])
        make_arrows(ax2, R_ri, arrows2)
        time_txt.set_text(f't = {t[i]:.2f}s')
        return arrows1 + arrows2 + [time_txt]

    ani = FuncAnimation(fig, update, frames=len(idx), interval=50, blit=False)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.close(fig)
    return ani


ani_track = animate_tracking(t_c, R_c, title='PID Tracking: Actual vs Reference')
print('Tracking animation created.')

In [ ]:
# Display tracking animation
HTML(ani_track.to_jshtml())

In [ ]:
# ============================================================
# GAIN STUDY: Effect of Kp, Kd, Ki on convergence
# ============================================================

gain_sets = [
    {'Kp': 1.0, 'Kd': 0.5, 'Ki': 0.1, 'label': 'Low gains'},
    {'Kp': 2.0, 'Kd': 1.5, 'Ki': 0.3, 'label': 'Medium gains (default)'},
    {'Kp': 4.0, 'Kd': 3.0, 'Ki': 0.5, 'label': 'High gains'},
]

fig, ax = plt.subplots(figsize=(12, 5))

for gs in gain_sets:
    # Patch gains
    global Kp, Kd, Ki
    Kp = gs['Kp']; Kd = gs['Kd']; Ki = gs['Ki']
    integral_eR = np.zeros(3)

    t_gs, R_gs, omega_gs, pi_gs = simulate(
        pid_controller,
        t_span=(0, T_END_ctrl),
        t_eval=t_ctrl,
        R0=Rz(np.radians(20)),
        pi0=np.zeros(3)
    )

    # Compute geodesic error
    errs = []
    for i, ti in enumerate(t_gs):
        R_r, *_ = reference_trajectory(ti)
        R_e_i   = R_r @ R_gs[i].T
        tr      = np.clip((np.trace(R_e_i)-1.0)/2.0, -1.0, 1.0)
        errs.append(np.degrees(np.arccos(tr)))

    ax.plot(t_gs, errs, linewidth=1.5, label=f"{gs['label']} (Kp={gs['Kp']}, Kd={gs['Kd']}, Ki={gs['Ki']})")

ax.set_xlabel('Time (s)')
ax.set_ylabel('Geodesic Error (degrees)')
ax.set_title('Effect of PID Gains on Tracking Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

# Restore default gains
Kp = 2.0; Kd = 1.5; Ki = 0.3

plt.tight_layout()
plt.savefig('/tmp/task3_gain_study.png', dpi=120, bbox_inches='tight')
plt.show()
print('Gain study complete.')

# Task #4 — Experimental Verification (Physical Lab)

Task #4 requires physical access to the twin rotor experimental setup in the Applied Mechanics Lab.

**Procedure:**
1. Implement the PID controller derived in Task #3 on the real-time hardware (e.g., dSPACE or Arduino-based controller)
2. Program the reference trajectory $R_r(t)$ (sinusoidal yaw trajectory as derived above)
3. Record actual $R(t)$ from encoders/IMU
4. Compare actual vs reference and compute the geodesic tracking error
5. Tune gains $K_P, K_D, K_I$ as needed based on observed performance
6. Verify stability: confirm the error converges to zero (or near-zero) as predicted by the simulation

**Expected behaviour (from simulation):**
- The orientation error should converge within ~5 seconds for the medium gain set
- Higher gains reduce rise time but may introduce overshoot or oscillations due to actuator limits
- The integral term eliminates steady-state error due to unmodelled disturbances (gravity, friction)

**References for real implementations on UAVs (similar controller):**
- https://youtu.be/6E9WDQNVSYA
- https://youtu.be/uUKxXImRMOA
- https://youtu.be/zq05N8m_9SA
- https://youtu.be/J5dThZGZN2g
- https://youtu.be/J5MMp6Be3tU
- https://youtu.be/6ZQgE1FI6Wc

## Summary

| Task | Description | Status |
|------|-------------|--------|
| Task 1 | Twin rotor dynamic model derivation on SO(3) | ✅ Completed |
| Task 2 | Simulation & animation (yaw spin + pitch oscillation) | ✅ Completed |
| Task 3 | PID tracking controller design & simulation | ✅ Completed |
| Task 4 | Experimental verification | 🔬 Requires physical lab access |